In [1]:
# Import thư viện
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
os.chdir('..')
import warnings

# Tắt cảnh báo
warnings.filterwarnings('ignore')
# Đổi style và set kích thước cho biểu đồ
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load the dataset
df = pd.read_csv('data/processed/03_outlier_cleaned.csv')
# Chuyển cột thời gian sang datetime
df['ISO_TIME'] = pd.to_datetime(df['ISO_TIME'])
# Sắp xếp dữ liệu theo SID và thời gian
df = df.sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)

# In thông tin cơ bản về dataset
print(f'Dataset: {df.shape[0]:,} dòng | {df["SID"].nunique():,} cơn bão')
print(f'Giai đoạn: {df["YEAR"].min()} - {df["YEAR"].max()}')

Dataset: 40,838 dòng | 719 cơn bão
Giai đoạn: 2000 - 2025


---
## 1. THỜI GIAN — SEASON

**Vấn đề:** Cột `MONTH` chỉ là con số 1–12, chưa thể hiện được đặc trưng mùa bão của khu vực nghiên cứu.

**Lý do:** Bão Việt Nam có tính mùa vụ rõ rệt. Nhóm thông tin này cần thiết cho phân tích xu hướng và mô hình hóa dữ liệu

**Giải pháp:** Tạo cột `SEASON` phân 4 nhóm theo đặc trưng khí hậu Việt Nam (JMA/NOAA):
- `Early` : tháng 6–8  — bão sớm, ít, thường yếu
- `Peak`  : tháng 9–10 — mùa bão chính, nhiều và mạnh nhất
- `Late`  : tháng 11–12 — bão muộn, ít nhưng hay đổ bộ miền Trung–Nam
- `Off`   : tháng 1–5  — ngoài mùa bão

**Nguồn tham khảo:** NOAA/NHC Western North Pacific Climatology; JMA Annual Report on Tropical Cyclones.


In [2]:
# Phân mùa bão theo đặc trưng khí hậu Việt Nam
season_map = {
    1: 'Off',
    2: 'Off',
    3: 'Off',
    4: 'Off',
    5: 'Off',
    6: 'Early',
    7: 'Early',
    8: 'Early',
    9: 'Peak',
    10: 'Peak',
    11: 'Late',
    12: 'Late'
}
df['SEASON'] = df['MONTH'].map(season_map)

print('Phân bố SEASON:')
print(df['SEASON'].value_counts().reindex(['Early', 'Peak', 'Late', 'Off']))
print(f"\nNaN trong SEASON: {df['SEASON'].isna().sum()} ")

Phân bố SEASON:
SEASON
Early    15828
Peak     12437
Late      7532
Off       5041
Name: count, dtype: int64

NaN trong SEASON: 0 


***Nhận xét:***

- Phân bố SEASON: Early (15,828) > Peak (12,437) > Late (7,532) > Off (5,041) — phù hợp với đặc trưng mùa bão Tây Bắc Thái Bình Dương, nhiều bão hình thành và di chuyển chậm trong tháng 7–8.
- Không có NaN (0 NaN) vì toàn bộ obs đều có MONTH.

---
## 2. PHÂN LOẠI CƯỜNG ĐỘ

Dữ liệu hiện tại chỉ có WMO_WIND (knots) và WMO_PRES (hPa) - là các con số thô, chưa thể biết ngay cơn bão đó mạnh hay yếu. Nhóm sẽ chuyển các con số đó thành nhãn phân loại TD / TS / TY phục vụ bài toán phân tích cường độ.

Ngoài ra, như đã xác định ở bước xử lý Missing Value, có 128 storms bị NaN toàn bộ track ở cả WMO_WIND lẫn WMO_PRES (không có điểm tham chiếu đầu hoặc cuối track). Thay vì xóa, nhóm giữ nguyên và đánh flag HAS_INTENSITY = False để việc phân loại cường độ tự động bỏ qua các storms này.

Các bước sẽ thực hiện, bổ sung thêm các cột sau:
- 2.1 INTENSITY_CAT       : phân loại từ WMO_WIND
- 2.2 INTENSITY_CAT_PRES  : phân loại từ WMO_PRES (để bù NaN)
- 2.3 INTENSITY_CAT_FINAL : kết hợp, ưu tiên WIND
- 2.4 CAT_CONFLICT        : kiểm tra mâu thuẫn giữa hai nguồn
- 2.5 PEAK_CAT_FINAL      : cấp đỉnh điểm của mỗi storm

## 2.1 INTENSITY_CAT — Phân loại cường độ từ WMO_WIND

**Vấn đề:** Do cột WMO_WIND chỉ có knots, chứ không đánh giá được cột đó là mạnh hay yếu.

**Lý do:** Để phục vụ bài toán phân tích cường độ, cần chuyển con số thô thành nhãn phân loại có ý nghĩa khí tượng học.

**Giải pháp:** Tạo cột `INTENSITY_CAT`mới để đánh giá bão thuộc loại nào, mạnh hay yếu, dựa trên thang WMO chuẩn cho khu vực Tây Bắc Thái Bình Dương (knot):
- TD  : ≤ 33 kt (Tropical Depression / Áp thấp nhiệt đới) - Yếu nhất
- TS  : 34–47 kt (Tropical Storm / Bão nhiệt đới)
- STS : 48–63 kt (Severe Tropical Storm / Bão nhiệt đới mạnh)
- TY  : ≥ 64 kt  (Typhoon / Bão) - Mạnh nhất

Ngoài ra, những dòng bão nào có WMO_WIND = NaN thì INTENSITY_CAT = NaN - do chưa đủ thông tin để phân loại cường độ

**Nguồn tham khảo:** ESCAP/WMO Typhoon Committee Operational Manual - Meteorological Component (2022 Edition).

In [3]:
# Phân loại cấp bão từ tốc độ gió
def classify_wind(wind):
    if pd.isna(wind):
        return np.nan
    elif wind <= 33:
        return 'TD'
    elif wind <= 47:
        return 'TS'
    elif wind <= 63:
        return 'STS'
    else:
        return 'TY'

df['INTENSITY_CAT'] = df['WMO_WIND'].apply(classify_wind)

print('Phân bố INTENSITY_CAT (từ WMO_WIND):')
print(df['INTENSITY_CAT'].value_counts(dropna=False))
print(f'\nSố lượng NaN: {df["INTENSITY_CAT"].isna().sum():,} obs')

Phân bố INTENSITY_CAT (từ WMO_WIND):
INTENSITY_CAT
NaN    18765
TY      7165
TS      7044
STS     4497
TD      3367
Name: count, dtype: int64

Số lượng NaN: 18,765 obs


**Nhận xét:**
- `INTENSITY_CAT` có **18,765 NaN** — gồm 4,117 obs của 128 storms không có intensity + 14,648 obs lẻ trong track của 591 storms HAS_INTENSITY=True mà WMO_WIND vẫn còn NaN sau bước xử lý nội suy, nên không thể phân loại cường độ.

- Phân bố hợp lý về mặt khí tượng: TS và TY chiếm đa số, STS ít hơn, TD ít nhất (TY 7,165 - TS 7,044 - STS 4,497 - TD 3,367). Điều này phù hợp với dữ liệu quan sát theo thời gian, khi nhiều cơn bão duy trì trạng thái TS và TY trong thời gian dài hơn so với các giai đoạn còn lại, vì đây là giai đoạn bão duy trì lâu nhất trong dòng đời bão. STS và TD ít hơn vì là giai đoạn chuyển tiếp ngắn (hình thành và tan)

## 2.2 INTENSITY_CAT_PRES — Phân loại cường độ từ WMO_PRES

**Vấn đề:** Sau bước 2.1, INTENSITY_CAT vẫn còn 18,765 NaN do WMO_WIND thiếu nhiều dữ liệu — chưa đủ để phân loại cường độ cho toàn bộ dataset.

**Lý do:** WMO_PRES (áp suất bão) có dữ liệu đầy đủ hơn WMO_WIND vì áp suất dễ đo hơn tốc độ gió. Hơn nữa, áp suất và cường độ bão có quan hệ nghịch chiều - áp suất càng thấp thì bão càng mạnh - nên có thể dùng để xấp xỉ cấp cường độ khi thiếu gió. Nên ta sẽ WMO_PRES như nguồn thông tin bổ sung.

**Giải pháp:** Tạo cột `INTENSITY_CAT_PRES` dựa trên ngưỡng Wind–Pressure Relationship cho khu vực Tây Bắc Thái Bình Dương, tương đồng với 4 thang cấp ở lớp 2.1:
- TD  : > 1000 hPa   (tương ứng ≤ 33 kt)
- TS  : 991–1000 hPa (tương ứng 34–47 kt)
- STS : 985–990 hPa  (tương ứng 48–63 kt)
- TY  : < 985 hPa    (tương ứng ≥ 64 kt)

**Lưu ý quan trọng:** Cột này này không thay thế phân loại chuẩn theo tốc độ gió, mà chỉ được dùng để hỗ trợ bù dữ liệu khi WMO_WIND bị thiếu (NaN)

**Nguồn tham khảo:** Atkinson, G. D., & Holliday, C. R. (1977). Tropical cyclone minimum sea level pressure / maximum sustained wind relationship for the western North Pacific. Monthly Weather Review.

In [4]:
def classify_pres(pres):
    if pd.isna(pres):
        return np.nan
    elif pres > 1000:
        return 'TD'
    elif pres >= 991:
        return 'TS'
    elif pres >= 985:
        return 'STS'
    else:
        return 'TY'

df['INTENSITY_CAT_PRES'] = df['WMO_PRES'].apply(classify_pres)

print('Phân bố INTENSITY_CAT_PRES (từ WMO_PRES):')
print(df['INTENSITY_CAT_PRES'].value_counts(dropna=False))
print(f'\nSố lượng NaN: {df["INTENSITY_CAT_PRES"].isna().sum():,} obs')

Phân bố INTENSITY_CAT_PRES (từ WMO_PRES):
INTENSITY_CAT_PRES
TS     10378
TY     10181
TD      8280
NaN     7848
STS     4151
Name: count, dtype: int64

Số lượng NaN: 7,848 obs


**Nhận xét:**
- `INTENSITY_CAT_PRES` có **7,848 NaN** — ít NaN hơn INTENSITY_CAT (18,765 NaN) vì WMO_PRES được ghi nhận đầy đủ hơn WMO_WIND trong IBTrACS.
- Cột này đóng vai trò **nguồn bù** — chỉ được dùng khi WMO_WIND = NaN. Không dùng thay thế hoàn toàn vì phân loại cường độ chuẩn vẫn dựa trên tốc độ gió, ngưỡng áp suất chỉ đang được tính gián tiếp.

## 2.3 INTENSITY_CAT_FINAL — Kết hợp, ưu tiên WIND

**Vấn đề:** Hiện có 2 cột phân loại cường độ (INTENSITY_CAT từ tốc độ gió và INTENSITY_CAT_PRES từ áp suất), cần kết hợp lại thành 1 cột duy nhất để sử dụng cho các bước phân tích tiếp theo.

**Lý do:** Tốc độ gió là tiêu chuẩn phân loại chính thức theo WMO, áp suất chỉ là thông tin gián tiếp - nên ưu tiên gió trước, chỉ dùng áp suất khi gió bị thiếu. Ưu tiên: WIND > PRES > NaN

**Giải pháp:** Tạo cột `INTENSITY_CAT_FINAL` theo thứ tự ưu tiên:
+ Nếu WMO_WIND có giá trị -> lấy từ INTENSITY_CAT
+ Nếu WMO_WIND = NaN      -> lấy từ INTENSITY_CAT_PRES
+ Nếu cả hai đều NaN      -> NaN

**Lưu ý:** Các storm có HAS_INTENSITY = False (128 storms NaN toàn track từ bước Missing Value) -> kết luận luôn là NaN (dù WMO_PRES đôi khi có giá trị), vì các storms này đã được xác định không có dữ liệu cường độ đáng tin cậy từ bước xử lý missing trước đó.

In [5]:
# Lấy cột INTENSITY_CAT làm gốc, chỗ nào NaN thì điền bằng giá trị tương ứng từ INTENSITY_CAT_PRES.
df['INTENSITY_CAT_FINAL'] = df['INTENSITY_CAT'].fillna(df['INTENSITY_CAT_PRES'])

# Lọc ra những hàng có HAS_INTENSITY = False và gán NaN cho toàn bộ những hàng đó.
# (Những dòng không có thông tin cường độ sẽ không được phân loại)
df.loc[~df['HAS_INTENSITY'], 'INTENSITY_CAT_FINAL'] = np.nan

print('Phân bố INTENSITY_CAT_FINAL:')
print(df['INTENSITY_CAT_FINAL'].value_counts(dropna=False))

# Tổng số dòng hợp lệ (có HAS_INTENSITY=True)
total_has = df[df['HAS_INTENSITY']].shape[0]
# Đếm số dòng hợp lệ HAS_INTENSITY=True đã phân loại được sau merge
filled = df[df['HAS_INTENSITY'] & df['INTENSITY_CAT_FINAL'].notna()].shape[0]
# Đếm số dòng lấy từ WIND
from_wind = (df['HAS_INTENSITY'] & df['INTENSITY_CAT'].notna()).sum()
# Đếm số dòng được PRES bổ sung
from_pres = (df['INTENSITY_CAT'].isna() & df['INTENSITY_CAT_PRES'].notna() & df['HAS_INTENSITY']).sum()
# Đếm số dòng HAS_INTENSITY=True nhưng sau merge vẫn NaN
still_nan = (df['HAS_INTENSITY'] & df['INTENSITY_CAT_FINAL'].isna()).sum()

# Coverage: Tỉ lệ dữ liệu đã phân loại thành công
print(f'\nCoverage phân loại cường độ: {filled:,}/{total_has:,} = {filled/total_has*100:.1f}%')
print(f'Từ WMO_WIND (nguồn chính) : {from_wind:,} obs')
print(f'Từ WMO_PRES (nguồn bù)    : {from_pres:,} obs')
print(f'Chưa phân loại được       : {still_nan:,} obs')

Phân bố INTENSITY_CAT_FINAL:
INTENSITY_CAT_FINAL
TS     10676
TD      9735
NaN     7832
TY      7607
STS     4988
Name: count, dtype: int64

Coverage phân loại cường độ: 33,006/36,721 = 89.9%
Từ WMO_WIND (nguồn chính) : 22,073 obs
Từ WMO_PRES (nguồn bù)    : 10,933 obs
Chưa phân loại được       : 3,715 obs


**Nhận xét:**
- Coverage đạt 89.9% (33,006/36,721 obs) trên các storms có `HAS_INTENSITY = True` - đạt mức ổn, chấp nhận được cho phân tích cường độ.
- Trong đó 22,073 obs lấy từ WMO_WIND và 10,933 obs được bù từ WMO_PRES - cho thấy áp suất đóng góp đáng kể vào việc tăng coverage.
- Còn 3,715 obs vẫn NaN do cả WMO_WIND lẫn WMO_PRES đều thiếu dữ liệu, không xử lý thêm vì không có nguồn thông tin nào đáng tin cậy để phân loại.
- 4,117 obs thuộc 128 storms `HAS_INTENSITY = False` được giữ NaN có chủ ý nhằm tránh suy luận cường độ cho các storms đã xác định không có dữ liệu đáng tin cậy từ bước Missing Value.

## 2.4 CAT_CONFLICT — Kiểm tra mâu thuẫn

Kiểm tra WIND và PRES có cho ra cùng cấp cường độ không.

**Vấn đề:** Đôi khi cùng một obs, phân loại từ gió cho ra "TS" nhưng phân loại từ áp suất lại cho ra "TY" — hai cái mâu thuẫn nhau.

**Lý do:** Hai thang đo được thiết kế độc lập, không có hiệu chỉnh chéo. Đặc biệt ở vùng ranh giới (ví dụ gió khoảng 47–48 knots — sát ngưỡng TS/STS, hoặc 63–64 knots — sát ngưỡng STS/TY), rất dễ bị lệch cấp.

**Giải pháp:** Flag lại những obs bị mâu thuẫn và ghi mức độ lệch
  + CAT_CONFLICT = True/False  (có mâu thuẫn không?)
  + CAT_CONFLICT_SEVERITY = 1  (lệch 1 cấp, VD: TD <=> TS, TS <=> STS, STS <=> TY)
  + CAT_CONFLICT_SEVERITY = 2  (lệch 2 cấp, VD: TD <=> STS, TS <=> TY)
  + CAT_CONFLICT_SEVERITY = 3  (lệch 3 cấp, VD: TD <=> TY)

In [6]:
# Chuyển cấp cường độ thành số để biểu diễn thứ tự mạnh yếu của bão (TY > STS > TS > TD)
cat_order = {'TD': 0, 'TS': 1, 'STS': 2, 'TY': 3}

# Tạo danh sách có đủ cả gió lẫn áp suất, chỉ những hàng này mới so sánh được
mask_both = df['INTENSITY_CAT'].notna() & df['INTENSITY_CAT_PRES'].notna()

# Tạo cột CAT_CONFLICT
df['CAT_CONFLICT'] = False
# Tạo cột severity (Mức độ lệch giữa phân loại từ WIND và phân loại từ PRES)
df['CAT_CONFLICT_SEVERITY'] = np.nan

# Kiểm tra conflict: Xem WIND và PRES có đang phân loại khác nhau không
df.loc[mask_both,'CAT_CONFLICT'] = (
    df.loc[mask_both,'INTENSITY_CAT'] != df.loc[mask_both, 'INTENSITY_CAT_PRES']
)
# Tính mức độ conflict: độ lệch số cấp giữa WIND và PRES (1 = lệch 1 cấp, 2 = lệch 2 cấp, 3 = lệch 3 cấp)
mask_conflict = (mask_both & df['CAT_CONFLICT'])

df.loc[mask_conflict,'CAT_CONFLICT_SEVERITY'] = (
    df.loc[mask_conflict,'INTENSITY_CAT'].map(cat_order) -
    df.loc[mask_conflict,'INTENSITY_CAT_PRES'].map(cat_order)
).abs()

# Đếm số conflict
n_conflict = df['CAT_CONFLICT'].sum()
# Tính phần trăm conflict
pct = n_conflict / mask_both.sum() * 100

print(f'Obs có đủ WIND + PRES để kiểm tra : {mask_both.sum():,}')
print(f'Obs phát hiện mâu thuẫn           : {n_conflict:,} ({pct:.2f}%)')

print('\nMức độ lệch phân loại (1=lệch 1 cấp, 2=lệch 2 cấp, 3=lệch 3 cấp):')
print(df[df['CAT_CONFLICT']]['CAT_CONFLICT_SEVERITY'].value_counts().sort_index())

print('\nBảng đối chiếu phân loại WIND vs PRES:')
conflict_df = df[df['CAT_CONFLICT']]

print(pd.crosstab(conflict_df['INTENSITY_CAT'], conflict_df['INTENSITY_CAT_PRES'],
                  rownames=['WIND'], colnames=['PRES']))

Obs có đủ WIND + PRES để kiểm tra : 22,057
Obs phát hiện mâu thuẫn           : 6,981 (31.65%)

Mức độ lệch phân loại (1=lệch 1 cấp, 2=lệch 2 cấp, 3=lệch 3 cấp):
CAT_CONFLICT_SEVERITY
1.0    6407
2.0     544
3.0      30
Name: count, dtype: int64

Bảng đối chiếu phân loại WIND vs PRES:
PRES   STS   TD    TS    TY
WIND                       
STS      0    1   344  2479
TD     436    0  1505    30
TS    1509  528     0   107
TY      42    0     0     0


**Nhận xét:**
- Conflict rate **31.65%** (6,981/22,057 obs), thoạt nhìn có vẻ cao, nhưng phần lớn là do boundary cases (gió nằm ở vùng ranh giới phân loại, ví dụ ~47–48 kt hoặc ~63–64 kt), nơi hai thang đo độc lập dễ cho kết quả lệch 1 cấp.
- Lệch 1 cấp chiếm đa số, tập trung ở các cặp ranh giới: TD <-> TS, TS <-> STS, STS <-> TY — đúng như kỳ vọng với thang 4 cấp mới.
- Lệch 2 cấp (VD: TD <-> STS, TS <-> TY) hiếm hơn đáng kể. Lệch 3 cấp (TD <-> TY) rất hiếm, nếu cần có thể kiểm tra thủ công.
- `CAT_CONFLICT` không ảnh hưởng đến kết quả phân loại cuối, chỉ đóng vai trò kiểm tra chất lượng dữ liệu - trong INTENSITY_CAT_FINAL, WMO_WIND đã được ưu tiên làm nguồn chính.

## 2.5 PEAK_CAT_FINAL — Cường độ đỉnh điểm của mỗi storm

**Vấn đề:** Một cơn bão có hàng trăm obs, mỗi obs có thể ở cấp khác nhau (lúc hình thành là TD, lúc mạnh nhất là TY, lúc tan là TS). Chúng ta cần biết cơn bão đó mạnh nhất là cấp mấy.

**Lý do:** Các bài toán phân tích xu hướng cường độ hoặc vẽ đường đi bão cần một nhãn duy nhất đại diện cho toàn bộ vòng đời của bão chứ  không thể dùng từng obs riêng lẻ.

**Giải pháp:** Lấy cấp cao nhất của mỗi storm từ `INTENSITY_CAT_FINAL` theo thứ tự TD < TS < STS < TY:
  + Nếu có bất kỳ obs nào là TY   -> PEAK_CAT_FINAL = "TY"
  + Nếu cao nhất chỉ là STS       -> PEAK_CAT_FINAL = "STS"
  + Nếu cao nhất chỉ là TS        -> PEAK_CAT_FINAL = "TS"
  + Nếu cao nhất chỉ là TD        -> PEAK_CAT_FINAL = "TD"
  + 128 storms không có intensity -> PEAK_CAT_FINAL = NaN

**Ý nghĩa:** Cột này là storm-level (một giá trị duy nhất cho mỗi storm), phục vụ trực tiếp cho:
- Vẽ đường đi bão - tô màu theo cấp đỉnh điểm
- Phân tích xu hướng - năm nào có nhiều TY nhất

In [7]:
# Max INTENSITY_CAT_FINAL per SID (TD < TS < STS < TY)
# Storm HAS_INTENSITY=False → NaN

cat_to_num = {'TD': 0, 'TS': 1, 'STS': 2, 'TY': 3}
num_to_cat = {0: 'TD', 1: 'TS', 2: 'STS', 3: 'TY'}

df['_cat_num'] = df['INTENSITY_CAT_FINAL'].map(cat_to_num)

peak_num = df.groupby('SID')['_cat_num'].max()
peak_cat = peak_num.map(num_to_cat)
df['PEAK_CAT_FINAL'] = df['SID'].map(peak_cat)

# Storm không có intensity → NaN
no_int_sids = df[~df['HAS_INTENSITY']]['SID'].unique()
df.loc[df['SID'].isin(no_int_sids), 'PEAK_CAT_FINAL'] = np.nan

df = df.drop(columns=['_cat_num'])

# Thống kê per storm
storm_peak = df.drop_duplicates('SID')['PEAK_CAT_FINAL']
total_has = storm_peak.notna().sum()

print('Phân bố PEAK_CAT_FINAL (per storm):')
print(storm_peak.value_counts(dropna=False))
print()
for cat in ['TY', 'STS', 'TS', 'TD']:
    n = (storm_peak == cat).sum()
    print(f'{cat} : {n} storms ({n/total_has*100:.1f}%)')
print(f'NaN: {storm_peak.isna().sum()} storms (HAS_INTENSITY=False)')

Phân bố PEAK_CAT_FINAL (per storm):
PEAK_CAT_FINAL
TY     258
TS     143
NaN    128
STS     97
TD      93
Name: count, dtype: int64

TY : 258 storms (43.7%)
STS : 97 storms (16.4%)
TS : 143 storms (24.2%)
TD : 93 storms (15.7%)
NaN: 128 storms (HAS_INTENSITY=False)


 **Nhận xét:**
- 591 storms có HAS_INTENSITY = True,được phân loại đỉnh điểm theo thang 4 cấp (TD < TS < STS < TY): TY 258 (43.7%), TS 143 (24.2%), STS 97 (16.4%), TD 93 (15.7%) - cho thấy phần lớn bão trong khu vực đạt cấp TS trở lên.
- 128 storms HAS_INTENSITY = False -> PEAK_CAT_FINAL = NaN có chủ ý, không suy luận cường độ khi không có dữ liệu đáng tin cậy.
- `PEAK_CAT_FINAL` là cột storm-level (một giá trị duy nhất per SID), phục vụ trực tiếp cho 2 bài toán chính:
  + Vẽ đường đi bão — tô màu đường theo cấp đỉnh điểm, vẽ đường đi có màu theo cấp bão
  + Phân tích xu hướng — thống kê số lượng TY/TS/STS/TD theo năm, thập kỷ

---
# TỔNG KẾT

In [8]:
new_cols = ['SEASON','INTENSITY_CAT','INTENSITY_CAT_PRES','INTENSITY_CAT_FINAL',
            'CAT_CONFLICT','CAT_CONFLICT_SEVERITY','PEAK_CAT_FINAL']

print('=== TỔNG KẾT FEATURE ENGINEERING ===')

print(f'Shape cuối dataset : {df.shape}')
print(f'Tổng feature mới   : {len(new_cols)}')

print('\nDanh sách feature mới:')
for c in new_cols:
    print(f'  - {c}')

print('\nSố lượng NaN trong feature mới:')
for c in new_cols:
    n_nan = df[c].isna().sum()
    pct   = n_nan / len(df) * 100
    print(f'  {c:30s}: {n_nan:,} NaN ({pct:.1f}%)')

output_path = 'data/final_data/final_dataset.csv'
df.to_csv(output_path, index=False)

=== TỔNG KẾT FEATURE ENGINEERING ===
Shape cuối dataset : (40838, 20)
Tổng feature mới   : 7

Danh sách feature mới:
  - SEASON
  - INTENSITY_CAT
  - INTENSITY_CAT_PRES
  - INTENSITY_CAT_FINAL
  - CAT_CONFLICT
  - CAT_CONFLICT_SEVERITY
  - PEAK_CAT_FINAL

Số lượng NaN trong feature mới:
  SEASON                        : 0 NaN (0.0%)
  INTENSITY_CAT                 : 18,765 NaN (45.9%)
  INTENSITY_CAT_PRES            : 7,848 NaN (19.2%)
  INTENSITY_CAT_FINAL           : 7,832 NaN (19.2%)
  CAT_CONFLICT                  : 0 NaN (0.0%)
  CAT_CONFLICT_SEVERITY         : 33,857 NaN (82.9%)
  PEAK_CAT_FINAL                : 4,117 NaN (10.1%)


**Nhận xét:**
- Dataset cuối có **40,838 obs, 20 cột** - thêm 7 cột mới so với bước Outlier.
- Engineering gồm: SEASON, INTENSITY_CAT, INTENSITY_CAT_PRES, INTENSITY_CAT_FINAL, CAT_CONFLICT, CAT_CONFLICT_SEVERITY và PEAK_CAT_FINAL.
- `SEASON = 0 NaN` - đúng vì năm được trích xuất đầy đủ từ thời gian quan trắc.
- `INTENSITY_CAT` (18,765 NaN) và `INTENSITY_CAT_PRES` (7,848 NaN) - sau khi kết hợp ở bước 2.3, `INTENSITY_CAT_FINAL` còn **7,832 NaN**, bao gồm 4,117 obs của 128 storms `HAS_INTENSITY=False` và 3,715 obs lẻ trong track mà cả hai nguồn đều thiếu.
- `CAT_CONFLICT = 0 NaN` - đúng vì cột này được khởi tạo mặc định `False` cho toàn bộ dataset, không có giá trị nào bị bỏ trống.
- `CAT_CONFLICT_SEVERITY` (33,857 NaN) — bình thường, vì cột này chỉ có giá trị khi obs có **đủ cả WIND lẫn PRES** và **có conflict**. Những obs còn lại giữ NaN có chủ ý.
- `PEAK_CAT_FINAL` (4,117 NaN) - đúng với 128 storms `HAS_INTENSITY=False`, mỗi storm trung bình ~32 obs.